# Name That LLM — Style Ensemble V2

Text-only, CPU, internet disabled, no external data. Character patterns and fixed writing-style measurements supply complementary evidence for each passage.

Development balanced accuracy: **35.6366%**. Second split with related prompts grouped: **36.1916%**, compared with **30.6714%** for the original baseline. Both use the available labeled development data and are not independent test scores.

The notebook fits on all labeled rows and generates one prediction per runtime test ID. It does not assume equal test class counts or fixed test size.

In [ ]:
import os
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "LOKY_MAX_CPU_COUNT"):
    os.environ[variable] = "2"


In [ ]:
"""Shared training and submission code, also embedded in the Kaggle notebook."""

from __future__ import annotations

import hashlib
import json
import logging
import platform
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import Normalizer
from sklearn.svm import LinearSVC

LOGGER = logging.getLogger("name_that_llm")
LABELS = tuple(sorted((
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra",
    "gpt-6-astra", "gemini-3.1-pro-preview", "gemini-3.6-flash",
    "gemini-3.7-flash", "gemini-3.8-flash", "claude-fable-5",
    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5",
    "claude-sonnet-5", "qwen3.8-max", "glm-5.3",
    "glm-5.3-flash", "kimi-k3",
)))


@dataclass(frozen=True)
class BaselineConfig:
    """Fixed before inspecting any validation results."""

    variant: str = "char"
    seed: int = 42
    n_splits: int = 5
    c: float = 1.0
    min_df: int = 2
    char_max_features: int = 250_000
    word_max_features: int = 100_000
    max_iter: int = 10_000
    tolerance: float = 1e-4


def validate_frame(frame: pd.DataFrame, *, training: bool) -> None:
    required = ["id", "text"] + (["prompt_id", "model"] if training else [])
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if frame.empty:
        raise ValueError("Input must contain at least one passage.")
    for column in required:
        if frame[column].isna().any() or not frame[column].map(
            lambda value: isinstance(value, str) and bool(value.strip())
        ).all():
            raise ValueError(f"Column {column!r} contains missing, empty or non-string values.")
    if frame["id"].duplicated().any():
        raise ValueError("Passage IDs must be unique.")
    if training and set(frame["model"]) != set(LABELS):
        raise ValueError("Training labels must match all 18 competition labels.")


def load_data(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Explicit string types retain leading zeros and the exact CSV-decoded text.
    train = pd.read_csv(data_dir / "train.csv", dtype=str, keep_default_na=False)
    test = pd.read_csv(data_dir / "test.csv", dtype=str, keep_default_na=False)
    validate_frame(train, training=True)
    validate_frame(test, training=False)
    return train, test


def build_pipeline(config: BaselineConfig) -> Pipeline:
    if config.variant not in {"char", "char_word"}:
        raise ValueError(f"Unknown baseline variant: {config.variant}")
    common = dict(min_df=config.min_df, sublinear_tf=True, dtype=np.float32)
    features = [("char", TfidfVectorizer(
        analyzer="char", ngram_range=(3, 5), lowercase=False,
        strip_accents=None, max_features=config.char_max_features, **common,
    ))]
    if config.variant == "char_word":
        features.append(("word", TfidfVectorizer(
            analyzer="word", ngram_range=(1, 2), lowercase=True,
            strip_accents=None, stop_words=None,
            max_features=config.word_max_features, **common,
        )))
    return Pipeline([
        ("features", FeatureUnion(features)),
        ("normalize", Normalizer()),
        ("classifier", LinearSVC(
            C=config.c, dual=True, random_state=config.seed,
            max_iter=config.max_iter, tol=config.tolerance,
        )),
    ])


def make_folds(train: pd.DataFrame, config: BaselineConfig) -> list[tuple[np.ndarray, np.ndarray]]:
    validate_frame(train, training=True)
    splitter = GroupKFold(
        n_splits=config.n_splits, shuffle=True, random_state=config.seed,
    )
    folds = list(splitter.split(train["text"], train["model"], groups=train["prompt_id"]))
    counts = np.zeros(len(train), dtype=int)
    for fitting, held_out in folds:
        if set(train.iloc[fitting]["prompt_id"]) & set(train.iloc[held_out]["prompt_id"]):
            raise ValueError("A prompt appears in both sides of a fold.")
        for indexes in (fitting, held_out):
            if set(train.iloc[indexes]["model"]) != set(LABELS):
                raise ValueError("Every fold must contain all 18 labels on both sides.")
        counts[held_out] += 1
    if not np.all(counts == 1):
        raise ValueError("Each passage must be held out exactly once.")
    return folds


def fit_checked(pipeline: Pipeline, text: pd.Series, labels: pd.Series) -> Pipeline:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        pipeline.fit(text, labels)
    for warning in caught:
        LOGGER.warning("%s: %s", warning.category.__name__, warning.message)
    if any(issubclass(w.category, ConvergenceWarning) for w in caught):
        raise RuntimeError("LinearSVC did not converge; inspect the run log before proceeding.")
    return pipeline


def validate_submission(
    test: pd.DataFrame, submission: pd.DataFrame,
) -> None:
    validate_frame(test, training=False)
    if list(submission.columns) != ["id", "model"]:
        raise ValueError("Submission columns must be exactly id, model.")
    if len(submission) != len(test):
        raise ValueError("Submission row count differs from the runtime test set.")
    if submission.isna().any().any() or submission["id"].duplicated().any():
        raise ValueError("Submission contains missing values or duplicate IDs.")
    if submission["id"].tolist() != test["id"].tolist():
        raise ValueError("Submission IDs must match runtime test IDs in order.")
    if not set(submission["model"]).issubset(LABELS):
        raise ValueError("Submission contains an unknown model label.")


def predict_submission(pipeline: Pipeline, test: pd.DataFrame) -> pd.DataFrame:
    validate_frame(test, training=False)
    predictions = pipeline.predict(test["text"])
    submission = pd.DataFrame({"id": test["id"].to_numpy(), "model": predictions})
    validate_submission(test, submission)
    return submission


def file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def runtime_versions() -> dict[str, str]:
    return {
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
    }



In [ ]:
"""Predeclared sparse-feature variants for the second experiment."""

from dataclasses import asdict, dataclass

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

import numpy as np


@dataclass(frozen=True)
class SparseConfig:
    name: str
    ngram_min: int
    ngram_max: int
    c: float
    min_df: int = 2
    max_features: int = 300_000
    seed: int = 42


SPARSE_CANDIDATES = (
    SparseConfig("char15_c4", 1, 5, 4.0),
    SparseConfig("char26_c4", 2, 6, 4.0),
    SparseConfig("char35_c4", 3, 5, 4.0),
    SparseConfig("char15_c16", 1, 5, 16.0),
)


def build_sparse_pipeline(config: SparseConfig) -> Pipeline:
    return Pipeline([
        ("features", TfidfVectorizer(
            analyzer="char", ngram_range=(config.ngram_min, config.ngram_max),
            lowercase=False, strip_accents=None, min_df=config.min_df,
            max_features=config.max_features, sublinear_tf=True, dtype=np.float32,
        )),
        ("classifier", LinearSVC(
            C=config.c, dual=True, random_state=config.seed,
            max_iter=10_000, tol=1e-4,
        )),
    ])


In [ ]:
"""Fixed, text-only style features for the separately recorded v2 experiment.

Feature families follow the character/word/function-word measurements in
https://arxiv.org/html/2104.08530v2, extended with Unicode punctuation and
position-specific discourse features. This is not a reproduction of that paper.
"""

from __future__ import annotations

from collections import Counter
import re
import string
import unicodedata

import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.svm import LinearSVC


# Fixed before fitting; no corpus vocabulary, labels, IDs, or prompt IDs.
FUNCTION_WORDS = tuple(dict.fromkeys("""
a an the and or but if because although though while whereas unless until when
whenever where wherever as than that whether since once before after during
despite through throughout within without with into onto upon from of for to
at by in on over under between among against about above below beyond across
around along toward towards per via not no nor never neither either both each
every any all some many much few little more most less least several enough
other another such same own only even also just still already yet again rather
quite very too so then thus hence therefore however instead otherwise indeed
perhaps maybe almost always often sometimes usually generally typically
especially particularly simply merely really actually essentially ultimately
is are was were be been being am do does did done doing have has had having
can could may might must shall should will would need ought
i me my mine myself we us our ours ourselves you your yours yourself yourselves
he him his himself she her hers herself it its itself they them their theirs
themselves one ones someone anyone everyone nobody nothing something everything
this these those there here who whom whose which what why how
""".split()))

CONNECTIVES = (
    "for example", "for instance", "in other words", "in practice",
    "in fact", "as a result", "on the other hand", "at the same time",
    "rather than", "instead of", "not only", "but also", "even if",
    "even though", "as long as", "so that", "such as", "in contrast",
    "by contrast", "in short", "ultimately", "however", "therefore",
    "because", "although", "while", "unless", "whether", "if",
)
PUNCTUATION = tuple(dict.fromkeys(string.punctuation + "—–‘’“”…−•\n\t"))
WORD_RE = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)*|\d+(?:\.\d+)?", re.UNICODE)
SENTENCE_RE = re.compile(r"(?<=[.!?])(?:[\"'”’)]*)\s+")


class StyleFeatures(TransformerMixin, BaseEstimator):
    """Deterministic measurements; fit deliberately learns nothing."""

    def fit(self, X, y=None):
        return self

    def __sklearn_is_fitted__(self):
        return True

    def get_feature_names_out(self, input_features=None):
        return np.asarray(list(self._measure("").keys()), dtype=object)

    def transform(self, X):
        if isinstance(X, str):
            raise ValueError("Expected a sequence of passages, not one string.")
        rows = [list(self._measure(text).values()) for text in X]
        if not rows:
            return np.empty((0, len(self.get_feature_names_out())), dtype=np.float64)
        matrix = np.asarray(rows, dtype=np.float64)
        if not np.isfinite(matrix).all():
            raise ValueError("Non-finite style feature.")
        return matrix

    @staticmethod
    def _measure(text):
        if not isinstance(text, str):
            raise TypeError("StyleFeatures accepts text strings only.")
        words = WORD_RE.findall(text.lower())
        counts = Counter(words)
        n = max(1, len(text))
        nw = max(1, len(words))
        lengths = np.asarray([len(w) for w in words] or [0], dtype=float)
        sentences = [s for s in SENTENCE_RE.split(text) if s.strip()]
        sentence_lengths = np.asarray(
            [len(WORD_RE.findall(s)) for s in sentences] or [0], dtype=float,
        )
        ns = max(1, len(sentences))
        f = {
            "log_chars": np.log1p(len(text)),
            "log_words": np.log1p(len(words)),
            "log_sentences": np.log1p(len(sentences)),
            "words_per_sentence": len(words) / ns,
            "chars_per_word": len(text) / nw,
            "type_token_ratio": len(counts) / nw,
            "hapax_ratio": sum(v == 1 for v in counts.values()) / nw,
            "repeated_token_ratio": sum(v for v in counts.values() if v > 1) / nw,
            "digits_rate": sum(c.isdigit() for c in text) / n,
            "uppercase_rate": sum(c.isupper() for c in text) / n,
            "letters_rate": sum(c.isalpha() for c in text) / n,
            "whitespace_rate": sum(c.isspace() for c in text) / n,
            "unicode_rate": sum(ord(c) > 127 for c in text) / n,
            "unicode_punctuation_rate": sum(
                ord(c) > 127 and unicodedata.category(c).startswith("P") for c in text
            ) / n,
            "contraction_rate": sum("'" in w or "’" in w for w in words) / nw,
            "short_word_rate": float(np.mean(lengths <= 3)),
            "long_word_rate": float(np.mean(lengths >= 8)),
            "question_sentence_rate": sum("?" in s for s in sentences) / ns,
            "exclamation_sentence_rate": sum("!" in s for s in sentences) / ns,
            "sentence_length_alternation": float(np.abs(np.diff(sentence_lengths)).mean())
            if len(sentence_lengths) > 1 else 0.,
        }
        for prefix, values in (("word_length", lengths), ("sentence_words", sentence_lengths)):
            for name, value in (
                ("mean", np.mean(values)), ("std", np.std(values)),
                ("min", np.min(values)), ("max", np.max(values)),
                ("q25", np.quantile(values, .25)), ("q50", np.quantile(values, .5)),
                ("q75", np.quantile(values, .75)),
            ):
                f[f"{prefix}_{name}"] = float(value)
        for length in range(1, 16):
            f[f"word_length_bin_{length}"] = float(np.mean(lengths == length))
        for char in PUNCTUATION:
            key = f"punct_u{ord(char):04x}"
            f[f"{key}_rate"] = text.count(char) / nw
            f[f"{key}_present"] = float(char in text)
        for char in string.ascii_lowercase:
            f[f"letter_{char}_rate"] = text.lower().count(char) / n
        for word in FUNCTION_WORDS:
            f[f"fw_{word}"] = counts[word] / nw
        lower = " " + " ".join(words) + " "
        first = " " + " ".join(words[:15]) + " "
        last = " " + " ".join(words[-15:]) + " "
        for connective in CONNECTIVES:
            needle = " " + connective + " "
            f[f"connective_{connective}"] = lower.count(needle) / nw
            f[f"opening_{connective}"] = float(needle in first)
            f[f"ending_{connective}"] = float(needle in last)
        for position, sentence in (
            ("opening", sentences[0] if sentences else ""),
            ("ending", sentences[-1] if sentences else ""),
        ):
            sw = WORD_RE.findall(sentence.lower())
            f[f"{position}_length_share"] = len(sw) / nw
            f[f"{position}_question"] = float("?" in sentence)
            f[f"{position}_colon"] = float(":" in sentence)
            f[f"{position}_dash"] = float("—" in sentence)
            f[f"{position}_first_person"] = sum(w in {"i", "we", "our", "my"} for w in sw) / max(1, len(sw))
            f[f"{position}_second_person"] = sum(w in {"you", "your"} for w in sw) / max(1, len(sw))
        return f


CANDIDATES = {
    "style_logistic": {
        "features": "fixed StyleFeatures -> StandardScaler",
        "classifier": "LogisticRegression", "C": .05,
        "solver": "lbfgs", "max_iter": 1500,
    },
    "style_hist": {
        "features": "fixed StyleFeatures", "classifier": "HistGradientBoostingClassifier",
        "max_iter": 160, "max_leaf_nodes": 7, "learning_rate": .07,
        "l2_regularization": 3., "min_samples_leaf": 25,
        "early_stopping": False, "random_state": 42,
    },
    "char_style_svm": {
        "features": "char TFIDF (3,5), min_df2, max_features250000, case-sensitive"
        " + .5 * L2-normalized standardized style; concatenate and L2-normalize",
        "classifier": "LinearSVC", "C": 1., "max_iter": 10000,
        "dual": True, "random_state": 42,
    },
}


def build_style_pipeline(name):
    style = StyleFeatures()
    if name == "style_logistic":
        return Pipeline([
            ("style", style), ("scale", StandardScaler()),
            ("classifier", LogisticRegression(C=.05, solver="lbfgs", max_iter=1500)),
        ])
    if name == "style_hist":
        return Pipeline([
            ("style", style),
            ("classifier", HistGradientBoostingClassifier(
                max_iter=160, max_leaf_nodes=7, learning_rate=.07,
                l2_regularization=3., min_samples_leaf=25,
                early_stopping=False, random_state=42,
            )),
        ])
    if name == "char_style_svm":
        return Pipeline([
            ("features", FeatureUnion([
                ("char", TfidfVectorizer(
                    analyzer="char", ngram_range=(3, 5), min_df=2,
                    max_features=250_000, lowercase=False, strip_accents=None,
                    sublinear_tf=True, dtype=np.float32,
                )),
                ("style", Pipeline([
                    ("extract", style), ("scale", StandardScaler()),
                    ("normalize", Normalizer()),
                ])),
            ], transformer_weights={"char": 1., "style": .5})),
            ("normalize", Normalizer()),
            ("classifier", LinearSVC(
                C=1., max_iter=10000, dual=True, random_state=42, tol=1e-4,
            )),
        ])
    raise ValueError(name)



In [ ]:
"""Small, predeclared count-based attribution models for improvement-v2.

The NB-SVM variant implements the log-count-ratio weighting in Wang and
Manning (2012), with character features and one-versus-rest multiclass fitting.
It does not use their optional coefficient interpolation. This is an adaptation,
not a reproduction of their sentiment experiment.
"""

from __future__ import annotations

import logging
import warnings
from dataclasses import dataclass

import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.utils.validation import check_is_fitted


LOGGER = logging.getLogger("name_that_llm.generative")


@dataclass(frozen=True)
class GenerativeConfig:
    """Three candidates are fixed before any improvement-v2 validation scores."""

    variant: str = "char_nbsvm"
    alpha: float = 1.0
    c: float = 1.0
    min_df: int = 2
    max_features: int = 250_000
    max_iter: int = 10_000
    tolerance: float = 1e-4
    seed: int = 42


GENERATIVE_CONFIGS = tuple(
    GenerativeConfig(variant=variant)
    for variant in ("char_mnb", "char_cnb", "char_nbsvm")
)


class GenerativeAttribution(ClassifierMixin, BaseEstimator):
    """Text-to-score estimator; only ``fit`` constructs a vocabulary.

    ``decision_function`` columns follow ``classes_`` (lexical label order).
    Scores are uncalibrated. NB scores are per-feature average log evidence;
    this preserves each row's predicted class while reducing length dependence.
    """

    def __init__(self, config: GenerativeConfig | None = None):
        self.config = config

    def fit(self, text, labels):
        config = self.config or GenerativeConfig()
        if config.variant not in {c.variant for c in GENERATIVE_CONFIGS}:
            raise ValueError(f"Unknown generative variant: {config.variant}")
        if config.alpha <= 0:
            raise ValueError("Smoothing must be strictly positive.")
        labels = np.asarray(labels)
        self.classes_ = np.unique(labels)
        if len(self.classes_) < 2:
            raise ValueError("At least two labels are required.")
        self.vectorizer_ = CountVectorizer(
            analyzer="char",
            ngram_range=(3, 5) if config.variant == "char_nbsvm" else (2, 5),
            lowercase=False, strip_accents=None, binary=True,
            min_df=config.min_df, max_features=config.max_features,
            dtype=np.float64,
        )
        counts = self.vectorizer_.fit_transform(text)
        self.variant_ = config.variant
        self.solver_iterations_ = []
        if config.variant == "char_mnb":
            self.classifier_ = MultinomialNB(alpha=config.alpha, fit_prior=False)
            self.classifier_.fit(counts, labels)
        elif config.variant == "char_cnb":
            self.classifier_ = ComplementNB(
                alpha=config.alpha, fit_prior=False, norm=True,
            )
            self.classifier_.fit(counts, labels)
        else:
            total = np.asarray(counts.sum(axis=0)).ravel()
            self.coef_ = np.empty((len(self.classes_), counts.shape[1]))
            self.intercept_ = np.empty(len(self.classes_))
            for index, label in enumerate(self.classes_):
                target = labels == label
                positive_count = np.asarray(counts[target].sum(axis=0)).ravel()
                positive = positive_count + config.alpha
                negative = total - positive_count + config.alpha
                ratio = np.log(positive / positive.sum()) - np.log(
                    negative / negative.sum()
                )
                weighted = counts.multiply(ratio).tocsr()
                classifier = LinearSVC(
                    C=config.c, dual=True, random_state=config.seed,
                    tol=config.tolerance, max_iter=config.max_iter,
                )
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    classifier.fit(weighted, target)
                for warning in caught:
                    LOGGER.warning("%s label=%s %s: %s", config.variant, label,
                                   warning.category.__name__, warning.message)
                if any(issubclass(w.category, ConvergenceWarning) for w in caught):
                    raise RuntimeError(f"NB-SVM did not converge for {label}.")
                # Fold the NB weighting into coefficients for cheap inference.
                self.coef_[index] = classifier.coef_[0] * ratio
                self.intercept_[index] = classifier.intercept_[0]
                self.solver_iterations_.append(int(classifier.n_iter_))
        self.n_features_in_ = counts.shape[1]
        return self

    def decision_function(self, text):
        check_is_fitted(self, ["classes_", "vectorizer_", "variant_"])
        counts = self.vectorizer_.transform(text)
        if self.variant_ == "char_nbsvm":
            scores = np.asarray(counts @ self.coef_.T) + self.intercept_
        else:
            scores = np.asarray(counts @ self.classifier_.feature_log_prob_.T)
            if self.variant_ == "char_mnb":
                scores += self.classifier_.class_log_prior_
            denominator = np.maximum(
                np.asarray(counts.sum(axis=1)).ravel(), 1.0,
            )
            scores /= denominator[:, None]
        if not np.isfinite(scores).all():
            raise ValueError("Model produced non-finite scores.")
        return scores

    def predict(self, text):
        return self.classes_[self.decision_function(text).argmax(axis=1)]


def build_generative(config: GenerativeConfig) -> GenerativeAttribution:
    return GenerativeAttribution(config)


In [ ]:
"""Fixed-weight combinations of text-only attribution evidence."""

import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted



def normalize_evidence(scores, *, probabilities=False):
    """Put different score types on a common per-passage scale, without fitting."""
    values = np.asarray(scores, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != len(LABELS) or not np.isfinite(values).all():
        raise ValueError("Expected finite passage-by-label evidence.")
    if probabilities:
        if np.any(values < 0) or not np.allclose(values.sum(axis=1), 1.0, atol=1e-5):
            raise ValueError("Invalid class probabilities.")
        values = np.log(np.clip(values, 1e-12, 1.0))
    centered = values - values.mean(axis=1, keepdims=True)
    return centered / np.maximum(centered.std(axis=1, keepdims=True), 1e-8)


def make_component(name):
    family, variant = name.split(":", 1)
    if family == "sparse":
        return build_sparse_pipeline(next(c for c in SPARSE_CANDIDATES if c.name == variant))
    if family == "style":
        return build_style_pipeline(variant)
    if family == "generative":
        return build_generative(GenerativeConfig(variant=variant))
    raise ValueError(f"Unknown component: {name}")


class EvidenceEnsemble(ClassifierMixin, BaseEstimator):
    def __init__(self, components, weights):
        self.components = components
        self.weights = weights

    def fit(self, X, y):
        weights = np.asarray(self.weights, dtype=float)
        if len(weights) != len(self.components) or not len(weights):
            raise ValueError("Each component needs a weight.")
        if not np.isfinite(weights).all() or (weights <= 0).any():
            raise ValueError("Weights must be positive and finite.")
        self.weights_ = weights / weights.sum()
        self.models_ = []
        self.classes_ = np.asarray(LABELS)
        for name in self.components:
            model = fit_checked(make_component(name), X, y)
            if tuple(model.classes_) != LABELS:
                raise ValueError("Component class order differs from competition labels.")
            self.models_.append(model)
        return self

    def decision_function(self, X):
        check_is_fitted(self, ["models_", "weights_", "classes_"])
        combined = None
        for model, weight in zip(self.models_, self.weights_):
            is_probability = hasattr(model, "predict_proba")
            raw = model.predict_proba(X) if is_probability else model.decision_function(X)
            evidence = normalize_evidence(raw, probabilities=is_probability)
            combined = weight * evidence if combined is None else combined + weight * evidence
        return combined

    def predict(self, X):
        return self.classes_[self.decision_function(X).argmax(axis=1)]


In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
recipe = json.loads('{"components": ["sparse:char15_c4", "style:style_hist"], "weights": [1, 1]}')
candidates = [
    Path("/kaggle/input/competitions/name-that-llm"),
    Path("/kaggle/input/name-that-llm"),
]
override = os.environ.get("NAME_THAT_LLM_DATA_DIR")
if override:
    candidates = [Path(override)]
data_dir = next((path for path in candidates if (path / "train.csv").is_file() and (path / "test.csv").is_file()), None)
if data_dir is None:
    raise FileNotFoundError("Attach the name-that-llm competition data.")
train, test = load_data(data_dir)
print(json.dumps({"versions": runtime_versions(), "recipe": recipe,
                  "train_rows": len(train), "test_rows": len(test)}, indent=2))
model = EvidenceEnsemble(**recipe).fit(train["text"], train["model"])
submission = predict_submission(model, test)
submission.to_parquet("submission.parquet", index=False)
validate_submission(test, pd.read_parquet("submission.parquet"))
print(f"Verified submission.parquet: {len(submission)} rows")
